# Clustering Analysis: K-means and PCA

**Project:** Public Compliance Data Analysis (MBA Thesis)  
**Author:** Enok  
**Last Updated:** 2026-02-10

---

## Objective

This notebook performs clustering analysis on Brazilian municipalities using socioeconomic indicators from the 2010 and 2022 Census data. The analysis includes:

1. **Descriptive Statistics** - Summary of the consolidated dataset
2. **PCA (Principal Component Analysis)** - Dimensionality reduction to identify key variance components
3. **K-means Clustering** - Group municipalities based on:
   - Population
   - Literacy rates
   - Income
   - Household social indicators
4. **Cluster Profiling** - Characterization of each cluster

---

## Data Source

- **Dataset:** `gold/consolidated_clustering/data.parquet`
- **Granularity:** One record per municipality (city)
- **Characteristics:**
  - No duplicate municipalities
  - No missing values in clustering features
  - Pre-normalized features (z-score standardization)
  - Data from both census years (2010 and 2022)

## 1. Setup and Data Loading

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# AWS
import boto3
import tempfile

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries loaded successfully!")

In [ ]:
# Configuration
BUCKET_NAME = "enok-mba-thesis-datalake"
DATA_KEY = "gold/consolidated_clustering/data.parquet"

# Load data from S3
def load_from_s3(bucket: str, key: str) -> pd.DataFrame:
    """Load parquet file from S3."""
    aws_profile = os.getenv("AWS_PROFILE", "mba-thesis")
    session = boto3.Session(profile_name=aws_profile)
    s3 = session.client('s3')
    with tempfile.NamedTemporaryFile(suffix='.parquet') as tmp:
        s3.download_file(bucket, key, tmp.name)
        return pd.read_parquet(tmp.name)

# Load the consolidated dataset
df = load_from_s3(BUCKET_NAME, DATA_KEY)
print(f"Loaded {len(df):,} municipalities")
print(f"Columns: {len(df.columns)}")

In [ ]:
# Display first rows
df.head()

In [ ]:
# Verify data quality
print("=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)
print(f"\nTotal municipalities: {len(df):,}")
print(f"Unique municipalities: {df['municipality_code'].nunique():,}")
print(f"Duplicate municipalities: {len(df) - df['municipality_code'].nunique()}")
print(f"\nMissing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values!")

## 2. Descriptive Statistics

In [ ]:
# Define raw feature columns (non-normalized)
raw_features = [
    'population_2010', 'population_2022', 'population_change_pct',
    'literacy_rate_2010', 'literacy_rate_2022', 'literacy_change_pp',
    'avg_income_2010', 'avg_income_2022', 'income_change_pct',
    'households_2010', 'households_2022', 'households_change_pct'
]

# Descriptive statistics
print("=" * 80)
print("DESCRIPTIVE STATISTICS - Raw Features")
print("=" * 80)
df[raw_features].describe().round(2).T

In [ ]:
# Distribution by region
print("\n" + "=" * 60)
print("DISTRIBUTION BY REGION")
print("=" * 60)

region_stats = df.groupby('region_name').agg({
    'municipality_code': 'count',
    'population_2022': ['sum', 'mean', 'median'],
    'literacy_rate_2022': 'mean',
    'avg_income_2022': 'mean'
}).round(2)

region_stats.columns = ['N_Municipalities', 'Total_Pop', 'Mean_Pop', 'Median_Pop', 
                        'Mean_Literacy', 'Mean_Income']
region_stats = region_stats.sort_values('N_Municipalities', ascending=False)
region_stats

In [ ]:
# Distribution by state
print("\n" + "=" * 60)
print("TOP 10 STATES BY NUMBER OF MUNICIPALITIES")
print("=" * 60)

state_counts = df.groupby(['state_name', 'region_name']).size().reset_index(name='n_municipalities')
state_counts = state_counts.sort_values('n_municipalities', ascending=False).head(10)
state_counts

In [ ]:
# Visualize distributions of key features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Population 2022 (log scale)
ax = axes[0, 0]
ax.hist(np.log10(df['population_2022']), bins=50, edgecolor='white', alpha=0.7)
ax.set_xlabel('Log10(Population 2022)')
ax.set_ylabel('Frequency')
ax.set_title('Population Distribution (2022)')

# Literacy Rate 2022
ax = axes[0, 1]
ax.hist(df['literacy_rate_2022'], bins=50, edgecolor='white', alpha=0.7, color='green')
ax.set_xlabel('Literacy Rate (%)')
ax.set_ylabel('Frequency')
ax.set_title('Literacy Rate Distribution (2022)')

# Income 2022
ax = axes[0, 2]
ax.hist(df['avg_income_2022'], bins=50, edgecolor='white', alpha=0.7, color='orange')
ax.set_xlabel('Average Income (BRL)')
ax.set_ylabel('Frequency')
ax.set_title('Income Distribution (2022)')

# Population Change
ax = axes[1, 0]
ax.hist(df['population_change_pct'], bins=50, edgecolor='white', alpha=0.7, color='purple')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Population Change (%)')
ax.set_ylabel('Frequency')
ax.set_title('Population Change (2010-2022)')

# Literacy Change
ax = axes[1, 1]
ax.hist(df['literacy_change_pp'], bins=50, edgecolor='white', alpha=0.7, color='teal')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Literacy Change (pp)')
ax.set_ylabel('Frequency')
ax.set_title('Literacy Change (2010-2022)')

# Income Change
ax = axes[1, 2]
ax.hist(df['income_change_pct'], bins=50, edgecolor='white', alpha=0.7, color='brown')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Income Change (%)')
ax.set_ylabel('Frequency')
ax.set_title('Income Change (2010-2022)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
plt.figure(figsize=(14, 10))
corr_matrix = df[raw_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Correlation Matrix - Socioeconomic Features', fontsize=14)
plt.tight_layout()
plt.show()

## 3. PCA - Principal Component Analysis

PCA will help us:
1. Reduce dimensionality while preserving variance
2. Identify which features contribute most to variance
3. Visualize municipalities in 2D/3D space
4. Potentially use fewer components for clustering

In [ ]:
# Use normalized features for PCA
norm_features = [f'{col}_norm' for col in raw_features]

# Extract normalized data
X_norm = df[norm_features].values

print(f"Feature matrix shape: {X_norm.shape}")
print(f"Number of features: {len(norm_features)}")

In [ ]:
# Perform PCA with all components
pca_full = PCA()
pca_full.fit(X_norm)

# Explained variance
explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# Display variance explained
print("=" * 60)
print("PCA - VARIANCE EXPLAINED")
print("=" * 60)
pca_df = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(len(explained_var))],
    'Variance_Explained': explained_var * 100,
    'Cumulative_Variance': cumulative_var * 100
})
print(pca_df.round(2).to_string(index=False))

In [ ]:
# Scree plot and cumulative variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
ax = axes[0]
components = range(1, len(explained_var) + 1)
ax.bar(components, explained_var * 100, alpha=0.7, label='Individual')
ax.plot(components, cumulative_var * 100, 'ro-', label='Cumulative')
ax.axhline(y=80, color='green', linestyle='--', label='80% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('PCA Scree Plot')
ax.legend()
ax.set_xticks(components)

# Cumulative variance
ax = axes[1]
ax.plot(components, cumulative_var * 100, 'b-o', linewidth=2, markersize=8)
ax.axhline(y=80, color='green', linestyle='--', label='80% threshold')
ax.axhline(y=90, color='orange', linestyle='--', label='90% threshold')
ax.axhline(y=95, color='red', linestyle='--', label='95% threshold')
ax.fill_between(components, cumulative_var * 100, alpha=0.3)
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Variance Explained (%)')
ax.set_title('Cumulative Variance Explained')
ax.legend()
ax.set_xticks(components)

plt.tight_layout()
plt.show()

# Determine optimal components
n_components_80 = np.argmax(cumulative_var >= 0.80) + 1
n_components_90 = np.argmax(cumulative_var >= 0.90) + 1
print(f"\nComponents needed for 80% variance: {n_components_80}")
print(f"Components needed for 90% variance: {n_components_90}")

In [ ]:
# PCA Loadings (feature contributions to each component)
loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=[f'PC{i+1}' for i in range(len(explained_var))],
    index=[col.replace('_norm', '') for col in norm_features]
)

print("=" * 60)
print("PCA LOADINGS (Feature Contributions)")
print("=" * 60)
print(loadings[['PC1', 'PC2', 'PC3', 'PC4']].round(3))

In [ ]:
# Visualize loadings heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(loadings[['PC1', 'PC2', 'PC3', 'PC4']], annot=True, cmap='RdBu_r', 
            center=0, fmt='.2f', linewidths=0.5)
plt.title('PCA Loadings - Feature Contributions to Principal Components', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Transform data to principal components
pca_3 = PCA(n_components=3)
X_pca = pca_3.fit_transform(X_norm)

# Add PCA components to dataframe
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]
df['PC3'] = X_pca[:, 2]

print(f"PCA transformation complete.")
print(f"Variance explained by 3 components: {pca_3.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# 2D PCA visualization by region
fig = px.scatter(
    df, x='PC1', y='PC2',
    color='region_name',
    hover_data=['municipality_name', 'state_name', 'population_2022', 'avg_income_2022'],
    title='PCA: Municipalities in 2D Space (by Region)',
    labels={'PC1': f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)'}
)
fig.update_layout(height=600)
fig.show()

In [ ]:
# 3D PCA visualization
fig = px.scatter_3d(
    df, x='PC1', y='PC2', z='PC3',
    color='region_name',
    hover_data=['municipality_name', 'state_name'],
    title='PCA: Municipalities in 3D Space (by Region)',
    labels={'PC1': f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)',
            'PC3': f'PC3 ({pca_3.explained_variance_ratio_[2]*100:.1f}%)'}
)
fig.update_layout(height=700)
fig.show()

## 4. K-means Clustering

We'll use K-means to group municipalities based on socioeconomic indicators:
- Population
- Literacy rates
- Income
- Household indicators

### 4.1 Determine Optimal Number of Clusters

In [ ]:
# Select features for clustering (using normalized features)
clustering_features = [
    'population_2022_norm',
    'literacy_rate_2022_norm',
    'avg_income_2022_norm',
    'households_2022_norm',
    'population_change_pct_norm',
    'literacy_change_pp_norm',
    'income_change_pct_norm',
    'households_change_pct_norm'
]

X_cluster = df[clustering_features].values
print(f"Clustering feature matrix: {X_cluster.shape}")
print(f"Features used: {clustering_features}")

In [ ]:
# Elbow method and Silhouette analysis
k_range = range(2, 11)
inertias = []
silhouettes = []

print("Evaluating K values...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster)
    inertias.append(kmeans.inertia_)
    sil_score = silhouette_score(X_cluster, kmeans.labels_)
    silhouettes.append(sil_score)
    print(f"  K={k}: Inertia={kmeans.inertia_:.0f}, Silhouette={sil_score:.4f}")

In [ ]:
# Plot Elbow and Silhouette
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
ax = axes[0]
ax.plot(list(k_range), inertias, 'b-o', linewidth=2, markersize=8)
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia (Within-cluster sum of squares)')
ax.set_title('Elbow Method')
ax.set_xticks(list(k_range))

# Silhouette plot
ax = axes[1]
ax.plot(list(k_range), silhouettes, 'g-o', linewidth=2, markersize=8)
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Analysis')
ax.set_xticks(list(k_range))

# Highlight best silhouette
best_k = list(k_range)[np.argmax(silhouettes)]
best_sil = max(silhouettes)
ax.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nBest K based on Silhouette Score: {best_k} (score={best_sil:.4f})")

### 4.2 Apply K-means with Optimal K

In [ ]:
# Use K based on analysis (adjust as needed)
OPTIMAL_K = best_k
print(f"Using K = {OPTIMAL_K} clusters")

# Fit final K-means model
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(X_cluster)

# Cluster distribution
print("\nCluster Distribution:")
print(df['cluster'].value_counts().sort_index())

In [ ]:
# Visualize clusters in PCA space (2D)
fig = px.scatter(
    df, x='PC1', y='PC2',
    color='cluster',
    color_continuous_scale='viridis',
    hover_data=['municipality_name', 'state_name', 'region_name', 'population_2022', 'avg_income_2022'],
    title=f'K-means Clusters (K={OPTIMAL_K}) in PCA Space',
    labels={'PC1': f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)',
            'cluster': 'Cluster'}
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(height=600)
fig.show()

In [ ]:
# Visualize clusters in 3D PCA space
fig = px.scatter_3d(
    df, x='PC1', y='PC2', z='PC3',
    color='cluster',
    color_continuous_scale='viridis',
    hover_data=['municipality_name', 'state_name'],
    title=f'K-means Clusters (K={OPTIMAL_K}) in 3D PCA Space'
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(height=700)
fig.show()

### 4.3 Cluster Profiling

In [ ]:
# Cluster statistics
print("=" * 80)
print("CLUSTER PROFILES - Mean Values")
print("=" * 80)

cluster_stats = df.groupby('cluster').agg({
    'municipality_code': 'count',
    'population_2022': ['mean', 'median'],
    'literacy_rate_2022': 'mean',
    'avg_income_2022': 'mean',
    'households_2022': 'mean',
    'population_change_pct': 'mean',
    'literacy_change_pp': 'mean',
    'income_change_pct': 'mean'
}).round(2)

cluster_stats.columns = ['N_Municipalities', 'Mean_Pop', 'Median_Pop', 'Mean_Literacy',
                         'Mean_Income', 'Mean_Households', 'Pop_Change', 'Lit_Change', 'Inc_Change']
cluster_stats

In [ ]:
# Cluster composition by region
print("\n" + "=" * 60)
print("CLUSTER COMPOSITION BY REGION")
print("=" * 60)

region_cluster = pd.crosstab(df['cluster'], df['region_name'], margins=True)
print(region_cluster)

In [ ]:
# Visualize cluster composition by region
fig = px.histogram(
    df, x='cluster', color='region_name',
    barmode='stack',
    title='Cluster Composition by Region',
    labels={'cluster': 'Cluster', 'count': 'Number of Municipalities'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Box plots for each feature by cluster
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

features_to_plot = [
    ('population_2022', 'Population 2022', True),
    ('literacy_rate_2022', 'Literacy Rate 2022 (%)', False),
    ('avg_income_2022', 'Avg Income 2022 (BRL)', False),
    ('households_2022', 'Households 2022', True),
    ('population_change_pct', 'Pop Change (%)', False),
    ('literacy_change_pp', 'Literacy Change (pp)', False),
    ('income_change_pct', 'Income Change (%)', False),
    ('households_change_pct', 'Households Change (%)', False)
]

for ax, (col, title, use_log) in zip(axes, features_to_plot):
    data = np.log10(df[col]) if use_log else df[col]
    ylabel = f'Log10({col})' if use_log else col
    df.boxplot(column=col if not use_log else None, by='cluster', ax=ax)
    if use_log:
        for i, cluster in enumerate(sorted(df['cluster'].unique())):
            cluster_data = np.log10(df[df['cluster'] == cluster][col])
            ax.boxplot(cluster_data, positions=[i+1])
    ax.set_title(title)
    ax.set_xlabel('Cluster')

plt.suptitle('Feature Distributions by Cluster', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Radar chart for cluster profiles
# Normalize cluster means for comparison
cluster_means = df.groupby('cluster')[raw_features].mean()
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

# Select key features for radar
radar_features = ['population_2022', 'literacy_rate_2022', 'avg_income_2022', 
                  'households_2022', 'population_change_pct', 'income_change_pct']

fig = go.Figure()

for cluster in sorted(df['cluster'].unique()):
    values = cluster_means_norm.loc[cluster, radar_features].values.tolist()
    values.append(values[0])  # Close the radar
    
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_features + [radar_features[0]],
        fill='toself',
        name=f'Cluster {cluster}'
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1])
    ),
    title='Cluster Profiles (Normalized)',
    height=600
)
fig.show()

### 4.4 Cluster Interpretation

In [ ]:
# Generate cluster interpretations based on statistics
print("=" * 80)
print("CLUSTER INTERPRETATION")
print("=" * 80)

for cluster in sorted(df['cluster'].unique()):
    cluster_data = df[df['cluster'] == cluster]
    n_muni = len(cluster_data)
    
    print(f"\n--- CLUSTER {cluster} ({n_muni} municipalities, {100*n_muni/len(df):.1f}%) ---")
    
    # Population
    pop_mean = cluster_data['population_2022'].mean()
    pop_median = cluster_data['population_2022'].median()
    pop_size = "Large" if pop_mean > df['population_2022'].mean() else "Small"
    print(f"  Population: {pop_size} (mean={pop_mean:,.0f}, median={pop_median:,.0f})")
    
    # Literacy
    lit_mean = cluster_data['literacy_rate_2022'].mean()
    lit_level = "High" if lit_mean > df['literacy_rate_2022'].mean() else "Low"
    print(f"  Literacy: {lit_level} ({lit_mean:.1f}%)")
    
    # Income
    inc_mean = cluster_data['avg_income_2022'].mean()
    inc_level = "High" if inc_mean > df['avg_income_2022'].mean() else "Low"
    print(f"  Income: {inc_level} (R$ {inc_mean:,.2f})")
    
    # Growth
    pop_change = cluster_data['population_change_pct'].mean()
    growth = "Growing" if pop_change > 0 else "Declining"
    print(f"  Population Trend: {growth} ({pop_change:+.1f}%)")
    
    # Top regions
    top_regions = cluster_data['region_name'].value_counts().head(2)
    print(f"  Main Regions: {', '.join(top_regions.index)}")

In [ ]:
# Example municipalities from each cluster
print("\n" + "=" * 80)
print("EXAMPLE MUNICIPALITIES FROM EACH CLUSTER")
print("=" * 80)

for cluster in sorted(df['cluster'].unique()):
    print(f"\n--- Cluster {cluster} ---")
    examples = df[df['cluster'] == cluster].nlargest(5, 'population_2022')[
        ['municipality_name', 'state_name', 'population_2022', 'avg_income_2022', 'literacy_rate_2022']
    ]
    print(examples.to_string(index=False))

## 5. Summary and Conclusions

In [ ]:
print("=" * 80)
print("ANALYSIS SUMMARY")
print("=" * 80)

print(f"""
DATASET:
  - Total municipalities analyzed: {len(df):,}
  - Features used: {len(raw_features)}
  - No missing values or duplicates

PCA RESULTS:
  - Components needed for 80% variance: {n_components_80}
  - Components needed for 90% variance: {n_components_90}
  - First 3 components explain: {pca_3.explained_variance_ratio_.sum()*100:.1f}% of variance

K-MEANS CLUSTERING:
  - Optimal K: {OPTIMAL_K} clusters
  - Silhouette Score: {silhouette_score(X_cluster, df['cluster']):.4f}
  - Cluster sizes: {dict(df['cluster'].value_counts().sort_index())}

KEY FINDINGS:
  - Municipalities can be grouped based on socioeconomic characteristics
  - Population size and income are major differentiating factors
  - Regional patterns are visible in cluster composition
  - Growth trajectories (2010-2022) vary significantly across clusters
""")

In [ ]:
# Save clustered data
output_columns = ['municipality_code', 'municipality_name', 'state_code', 'state_name',
                  'region_code', 'region_name', 'cluster', 'PC1', 'PC2', 'PC3'] + raw_features

df_output = df[output_columns].copy()
print(f"Output dataset ready with {len(df_output)} rows and {len(output_columns)} columns")
df_output.head()

In [ ]:
# Save to S3 (optional)
# output_key = 'gold/clustered_municipalities/data.parquet'
# with tempfile.NamedTemporaryFile(suffix='.parquet') as tmp:
#     df_output.to_parquet(tmp.name, index=False)
#     aws_profile = os.getenv("AWS_PROFILE", "mba-thesis")
#     session = boto3.Session(profile_name=aws_profile)
#     s3 = session.client('s3')
#     s3.upload_file(tmp.name, BUCKET_NAME, output_key)
#     print(f"Saved to s3://{BUCKET_NAME}/{output_key}")

---

## End of Analysis

This notebook demonstrated:
1. Loading and validating the consolidated municipality dataset
2. Descriptive statistics and feature distributions
3. PCA for dimensionality reduction (identifying key variance components)
4. K-means clustering to group municipalities by socioeconomic characteristics
5. Cluster profiling and interpretation

**Next Steps:**
- Use cluster labels for stratified analysis
- Investigate compliance patterns within each cluster
- Compare cluster characteristics with sanctions data